<a href="https://colab.research.google.com/github/SebaAcunaC/YOLO11-Vigilancia-Urbana-USACH/blob/main/Semana-2-Entrenamiento/Semana2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==============================================================================
# PROYECTO DE FIN DE GRADO - USACH
# Master Notebook: Benchmarking Multimodelo y Optimización Multiresolución
# Autor: Sebastian [Apellido]
# ==============================================================================

# 1. SETUP DE REPRODUCIBILIDAD
# Instalamos Ultralytics para YOLO y Torchvision para modelos Legacy (SSD/R-CNN)
!pip install ultralytics roboflow torchvision -q

import os
import yaml
import torch
import time
import pandas as pd
from ultralytics import YOLO
from roboflow import Roboflow
from torchvision.models.detection import ssd300_vgg16, fasterrcnn_resnet50_fpn

# 2. CARGA DE DATASET (RECONOCIMIENTO DE PERSONAS)
# Requerimiento: Mismo dataset para todos los modelos
api_key = input("Introduce tu API Key de Roboflow: ")
rf = Roboflow(api_key=api_key)
project = rf.workspace("sebastianacua").project("person-hgivm-vyoba")
dataset = project.version(1).download("yolov8")

# 3. CARGA DE MODELOS (COMPARATIVA SOLICITADA POR EL PROFESOR)
# Cargamos YOLO11 (Tu optimizado), YOLOv8 (Base) y Modelos Legacy
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Hardware detectado: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

model_yolo11 = YOLO("yolo11n.pt") # Estado del arte
model_yolov8 = YOLO("yolov8n.pt") # Versión anterior
# SSD y Faster R-CNN (Modelos Legacy requeridos)
model_ssd = ssd300_vgg16(pretrained=True).to(device).eval()
model_rcnn = fasterrcnn_resnet50_fpn(pretrained=True).to(device).eval()

# 4. SCRIPT DE BENCHMARKING MULTI-RESOLUCIÓN
# Aplicamos el barrido de la Semana 5 para justificar el 'Sweet Spot'
def benchmark_model(model_obj, resolutions=[1088, 640, 320]):
    results = []
    for res in resolutions:
        # Nota: Usamos Stride 32 para optimización de memoria en GPU
        print(f"Evaluando en resolución: {res}p...")
        start_time = time.time()
        # Inferencia en modo stream para proteger la RAM
        metrics = model_obj.predict(source=dataset.location + "/valid/images", imgsz=res, device=device, conf=0.5)

        # Cálculo de métricas de ingeniería
        avg_inference = sum([m.speed['inference'] for m in metrics]) / len(metrics)
        fps = 1000 / avg_inference
        results.append({"Res": res, "FPS": round(fps, 2), "Inference_ms": round(avg_inference, 2)})
    return results

# 5. EJECUCIÓN Y TABLA DE RESULTADOS
print("\nIniciando Comparativa Maestra...")
res_yolo11 = benchmark_model(model_yolo11)
# Aquí puedes agregar el benchmark de los otros modelos para completar tus tablas

# Resumen Final para el Mini-Paper
df_final = pd.DataFrame(res_yolo11)
print("\n--- RESULTADOS FINALES YOLO11 ---")
print(df_final)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.8/95.8 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 41.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 47.0 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Introduce tu API Key de Roboflow: Zz5RMlP0mViMO19NbtEq
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to person-1 in yolov8:: 100%|██████████| 1145/1145 [00:00<00:00, 5554.26it/s]


Hardware detectado: CPU


The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=SSD300_VGG16_Weights.COCO_V1`. You can also use `weights=SSD300_VGG16_Weights.DEFAULT` to get the most up-to-date weights.


Downloading: "https://download.pytorch.org/models/ssd300_vgg16_coco-b556d3b4.pth" to /root/.cache/torch/hub/checkpoints/ssd300_vgg16_coco-b556d3b4.pth


100%|██████████| 136M/136M [00:03<00:00, 44.3MB/s]
Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=FasterRCNN_ResNet50_FPN_Weights.COCO_V1`. You can also use `weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT` to get the most up-to-date weights.


Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth" to /root/.cache/torch/hub/checkpoints/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth


100%|██████████| 160M/160M [00:01<00:00, 89.9MB/s]



Iniciando Comparativa Maestra...
Evaluando en resolución: 1088p...

image 1/110 /content/person-1/valid/images/-_10_jpeg.rf.a655070e34061384ae7dca75f0480932.jpg: 768x1088 3 persons, 513.9ms
image 2/110 /content/person-1/valid/images/-_19_jpeg.rf.97f0e75cee8f7929bbf1c318302faeaa.jpg: 736x1088 5 persons, 1 bicycle, 1 car, 417.6ms
image 3/110 /content/person-1/valid/images/-_22_jpeg.rf.54a92cc2551d6be9396d64428b801f72.jpg: 736x1088 5 persons, 419.3ms
image 4/110 /content/person-1/valid/images/-_30_jpeg.rf.ef1355d33d2780a2b9951a87ec66b489.jpg: 640x1088 5 persons, 8 cars, 1 handbag, 366.1ms
image 5/110 /content/person-1/valid/images/-_31_jpeg.rf.ea80eb4433157bc65602a35ac7b89239.jpg: 736x1088 7 persons, 1 bicycle, 2 cars, 1 truck, 1 traffic light, 1 backpack, 3 handbags, 404.7ms
image 6/110 /content/person-1/valid/images/-_32_jpeg.rf.f869a8879cc0e788956dd45594979276.jpg: 736x1088 5 persons, 1 car, 1 umbrella, 390.6ms
image 7/110 /content/person-1/valid/images/-_33_jpeg.rf.c0b641c89b152ff30d